In [1]:
from langchain_openai import ChatOpenAI
from langchain.agents.format_scratchpad.openai_tools import (
    format_to_openai_tool_messages,
)
from langchain.agents.output_parsers.openai_tools import OpenAIToolsAgentOutputParser
from langchain.agents import AgentExecutor
from langchain_core.messages import AIMessage, HumanMessage
from tools.material_extraction import (
    get_material_ids,
    MPSurfacePropertiesTool,
    MPPhononTool,
    MPElasticityTool,
    MPThermoTool,
    MPDielectricTool,
    MPPiezoelectricTool,
    MPMagnetismTool,
    MPSynthesisTool,
    MPElectronicStructureTool,
    MPElectronicBandStructureTool,
    MPElectronicDensityOfStatesTool,
    MPOxidationStatesTool,
    MPBondsTool,
    MPAbsorptionTool,
    extract_materials_matweb
)
from tools.material_extraction import OPENAI_API

In [2]:
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0, api_key=OPENAI_API)

tools = [
    get_material_ids,
    MPSurfacePropertiesTool(),
    MPPhononTool(),
    MPElasticityTool(),
    MPThermoTool(),
    MPDielectricTool(),
    MPPiezoelectricTool(),
    MPMagnetismTool(),
    MPSynthesisTool(),
    MPElectronicStructureTool(),
    MPElectronicBandStructureTool(),
    MPElectronicDensityOfStatesTool(),
    MPOxidationStatesTool(),
    MPBondsTool(),
    MPAbsorptionTool(),
    extract_materials_matweb
]
MEMORY_KEY = "chat_history"
from langchain.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate, MessagesPlaceholder

system_message = SystemMessagePromptTemplate.from_template(
    """
    You are a powerful agent for performing data extraction from the Materials Project and matweb.com.
    Your task is to determine the type of problem in the user's query before taking action. 
    First, classify the problem into one of the following types:
    1. Finding materials based on given physical properties (e.g., Young's modulus, tensile strength).
    2. Getting properties of a specific material (e.g., getting the properties of mp-149).
    
    Based on your classification, proceed with the appropriate action:
    - If it's Type 1 (Finding materials based on properties), explain the tool you are using (e.g., searching for materials based on physical properties), what parameters you are using, and why you chose those parameters. Then, execute the appropriate action to search for materials.
    - If it's Type 2 (Getting properties of a specific material), first find the material_id(s) for the material, then explain the tool you are using (e.g., fetching material properties), the parameters, and the reasoning behind those choices. Then, execute the appropriate action to retrieve the properties of the specific material.
    
    DO NOT MAKE YOUR OWN DATA UP. If Materials Project does not return any results, state that and ask for more information.
    After running the extraction from materials project, if the problem is type 1, extract from matweb as well if the property is
     relevant to the agent
    """
)

# human message template
human_message = HumanMessagePromptTemplate.from_template(
    """
    Given the user's query: '{input}', first classify the problem as either '1' for finding materials
 based on physical properties or '2' for getting material properties. If it is '2', then state the chemical
 formula of the  specified materials and its contituent elements seperately. Use the chemical formula to get material_id(s)
 first then proceed. If it is type '1' use the appropiate tool.
    """
)

prompt = ChatPromptTemplate.from_messages(
    [
        system_message,
        MessagesPlaceholder(variable_name=MEMORY_KEY),
        human_message,
        MessagesPlaceholder(variable_name="agent_scratchpad"),
    ]
)

llm_with_tools = llm.bind_tools(tools)

agent = (
    {
        "input": lambda x: x["input"],
        "agent_scratchpad": lambda x: format_to_openai_tool_messages(
            x["intermediate_steps"]
        ),
        "chat_history": lambda x: x["chat_history"],
    }
    | prompt
    | llm_with_tools
    | OpenAIToolsAgentOutputParser()
)

agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

In [9]:
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)
chat_history = []

input1 = """
Identify and extract materials with a specific heat capacity in the range of 0.1 to 2.0 Joules per gram per degree
 Celsius from the dataset. Ensure the retrieved materials represent diverse material classes, including metals, polymers,
 ceramics, and composites. For each material, provide its specific heat capacity value along with additional thermal,
 mechanical, and compositional properties, as available. Prioritize entries with experimentally measured data, clearly
 indicating the temperature conditions under which the specific heat capacity was determined. Exclude materials with
 estimated or incomplete data, and present the results in a structured format for comparative analysis.
"""
result = agent_executor.invoke({"input": input1, "chat_history": chat_history})
chat_history.extend(
    [
        HumanMessage(content=input1),
        AIMessage(content=result["output"]),
    ]
)



> Entering new AgentExecutor chain...

Invoking: `extract_materials_matweb` with `{'property': 'Specific Heat Capacity', 'low': 0.1, 'high': 2.0, 'num_materials': 5}`
responded: This problem falls under Type 1: Finding materials based on physical properties. To address this, I will use the `extract_materials_matweb` tool to identify and extract materials with a specific heat capacity in the range of 0.1 to 2.0 Joules per gram per degree Celsius from the dataset. I will ensure that the retrieved materials represent diverse material classes, including metals, polymers, ceramics, and composites. Additionally, I will provide additional thermal, mechanical, and compositional properties for each material, prioritizing entries with experimentally measured data.

Let's proceed with extracting the materials based on the specified range of specific heat capacity.

Number of results: 11245


The material analyzed is Silver (Ag), with key properties including a density of 10.491 g/cc, an atomic mass of 107.8682, and an atomic number of 47. Silver shows an electrode potential of 0.799 V and an electronegativity of 1.93. Other significant characteristics involve a Vickers hardness of 25 when annealed, a tensile strength of 140 MPa in the annealed state, and a modulus of elasticity of 76.0 GPa. Silver possesses a linear coefficient of thermal expansion of 19.6 μm/m-°C from 20.0 - 100 °C and a melting point of 961.93 °C. Its thermal conductivity is measured at 419 W/m-K, while the heat of fusion is 105 J/g and the heat of vaporization is 2332 J/g. Furthermore, it demonstrates an electrical resistivity of 0.00000155 ohm-cm and a shear modulus of 27.8 GPa when annealed. Silver's properties also include a specific heat capacity of 0.234 J/g-°C and a boiling point of 2162 °C. Additionally, the material exhibits an emissivity of 0.055 at 700 °C and a visible reflection coefficient o